# 01 — Dataset overview and reproducible holdouts
Run cells from top to bottom using the Chemtellect kernel. Begin with `chemicalspace_new/package_00`.

This notebook audits completed local files only. It reuses your existing validators, keeps package populations separate, and never edits raw files. No fingerprint generation or retrieval benchmark runs yet.

**Outputs:** fresh audit report and a saved split manifest. For either 100K parent, one saved set of 100 queries and one ordered 100K index support all three nested sizes.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display
from chemtellect.dataset_overview import (
    inventory, dataset, audit, make_split, index_records, descriptor_rows, save_json,
)

# Find the repository whether the kernel starts in its root or notebooks directory.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file() and (p / 'src/chemtellect').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Start JupyterLab inside your Chemtellect repository.')
print('Repository:', ROOT)
print('Interpreter:', sys.executable)

Repository: /home/ruben/Coding Projects/chemtellect
Interpreter: /home/ruben/Coding Projects/chemtellect/.venv/bin/python3


## 1. Inventory
Presence is not validation. Only the nine approved pools are listed; partial, archived, smoke and 1M files are excluded.

In [2]:
display(pd.DataFrame(inventory(ROOT)))

,source,package,raw_present,manifest_present,report_present,bytes
0,chemicalspace_new,package_00,True,True,True,147762
1,chemicalspace_new,package_01,True,True,True,148259
2,chemicalspace_new,package_02,True,True,True,147746
3,chemicalspace_new,scale_10k_00,True,True,True,1293222
4,chemicalspace_new,scale_100k_00,True,True,True,12555228
5,otava_chemical,package_00,True,True,True,228476
6,otava_chemical,package_01,True,True,True,227796
7,otava_chemical,package_02,True,True,True,227988
8,otava_chemical,scale_100k_00,True,True,True,19443748


## 2. Select one pool
Change only `SOURCE` and `PACKAGE` below when moving to another run. The configuration supplies the intended sizes. The standalone 10K pool is not a subset of a 100K parent.

In [3]:
config = json.loads((ROOT / 'experiments/configs/dataset_overview_v1.json').read_text())
SOURCE = config['source']
PACKAGE = config['package']
item = dataset(ROOT, SOURCE, PACKAGE)
INDEX_SIZES = config['index_sizes_by_package'][PACKAGE]
print(SOURCE, PACKAGE, 'index sizes:', INDEX_SIZES)

chemicalspace_new package_00 index sizes: [1000]


## 3. Verify provenance and recompute molecular validation
Checks manifest identity, raw hash, counts and source-ID integrity. Fresh chemistry checks use the existing canonical-isomeric-SMILES validator without salt, tautomer or protonation standardization.

A historical report with a mismatched hash, policy or count stops execution. An absent report permits fresh validation. A changed RDKit version is reported explicitly. RDKit diagnostic warning lines are not equivalent to rejected-record counts.

In [4]:
summary, records = audit(item)
if summary['counts']['total_input_records'] != config['expected_raw_counts'][PACKAGE]:
    raise ValueError('Raw count differs from the approved experiment inventory')
print('Raw SHA-256:', summary['sha256'])
print('Historical report:', summary['historical_report_status'])
print('RDKit:', summary['rdkit_version'])
display(pd.Series(summary['counts'], name='count').to_frame())
display(pd.DataFrame(records[:5]))

Raw SHA-256: f3b298720497eb4ada39fa0dcfdf85f2c5851922ea8d0423c6d68d45f45c0f39
Historical report: matched
RDKit: 2026.09.1


,count
total_input_records,1200
successfully_parsed_records,1200
invalid_molecular_records,0
integrity_rejected_records,0
unique_canonical_structures,1200
duplicate_structure_groups,0
records_in_duplicate_groups,0
surplus_duplicate_records,0


,source_id,line_number,smiles,canonical_smiles
0,objectid:5be035a2f12700469a512d77,1,CNCCCNC(=O)Cn1cc(S(=O)(=O)c2ccc(F)cc2)c(=O)c2c...,CNCCCNC(=O)Cn1cc(S(=O)(=O)c2ccc(F)cc2)c(=O)c2c...
1,objectid:5be4f200f127006b848928c3,2,CCOc1ccc2c(c1)c(=O)c(-c1nc(-c3cccc(Cl)c3)no1)c...,CCOc1ccc2c(c1)c(=O)c(-c1nc(-c3cccc(Cl)c3)no1)c...
2,objectid:5bdd3865f1270023c1caf2ae,3,NCCCN(c1ccccc1)S(=O)(=O)C1CCOCC1,NCCCN(c1ccccc1)S(=O)(=O)C1CCOCC1
3,objectid:5bc959e3f12700529cb4314b,4,CC1CCC(C(C)C)C2(CCC(CNC(C)(C)C)O2)C1,CC1CCC(C(C)C)C2(CCC(CNC(C)(C)C)O2)C1
4,objectid:5bdce5edf1270023c1a65fdf,5,CCCn1cc(C(=O)c2cccc3c2CCN(C(=O)OC(C)(C)C)C3)cn1,CCCn1cc(C(=O)c2cccc3c2CCN(C(=O)OC(C)(C)C)C3)cn1


## 4. Construct and inspect the holdout
Keep the first physical raw line for each canonical structure. Rank representatives using SHA-256 of a versioned encoding of seed, source, package and source ID; select the first 100 as queries. The next rows form the index ordering. This seed-42 algorithm is deterministic and does not depend on a NumPy RNG version.

Persist the actual records and ordering, not just the seed. Changing the seed, canonicalization version or input changes experiment identity. No molecule in the query set can share its canonical structure with an index molecule. Different structures can still share fingerprints later.

In [5]:
split = make_split(summary, records, INDEX_SIZES, config['query_count'], config['seed'])
query_structures = {r['canonical_smiles'] for r in split['queries']}
assert query_structures.isdisjoint(r['canonical_smiles'] for r in split['index_order'])
for size in INDEX_SIZES:
    assert len(index_records(split, size)) == size
print('Split ID:', split['split_id'])
print('Queries:', len(split['queries']), 'Largest index:', len(split['index_order']),
      'Unused valid structures:', split['unused_count'])
display(pd.DataFrame([{'index_size': size, 'queries': len(split['queries'])} for size in INDEX_SIZES]))

Split ID: f85552ea5b17e4829a09df82c79ad3bb2d762ee672c0f52de234dc7fc668e908
Queries: 100 Largest index: 1000 Unused valid structures: 100


,index_size,queries
0,1000,100


## 5. Descriptor overview
For responsiveness, describe at most 5,000 eligible unique structures, selected by source-ID ordering. This descriptive subset is not asserted to be representative. Set `descriptor_limit` to `null` to describe the entire selected pool. It is separate from the holdout and index membership; descriptors do not change the split.

In [6]:
limit = config['descriptor_limit']
descriptor_input = sorted(records, key=lambda r: r['source_id'])
if limit is not None:
    if type(limit) is not int or limit <= 0:
        raise ValueError('descriptor_limit must be null or a positive integer')
    descriptor_input = descriptor_input[:limit]
descriptors = pd.DataFrame(descriptor_rows(descriptor_input))
print(f'Describing {len(descriptors):,} of {len(records):,} eligible unique structures')
display(descriptors.drop(columns='source_id').describe())

Describing 1,200 of 1,200 eligible unique structures


,molecular_weight,logp,tpsa,heavy_atoms,rotatable_bonds
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,323.399694,2.726665,63.035567,22.089167,4.927500
std,100.924725,1.523909,27.031068,7.016213,2.420839
min,113.204000,-1.579000,0.000000,8.000000,0.000000
25%,256.348250,1.677755,43.090000,17.000000,3.000000
50%,301.238500,2.710260,61.440000,20.000000,5.000000
75%,365.337750,3.588640,79.207500,25.000000,6.000000
max,842.965000,8.332900,213.950000,60.000000,17.000000


## 6. Save derived artifacts
These folders are already intended to be ignored by Git. Existing identical JSON can be reused; a conflicting file is never overwritten. Save the audit separately from the original acquisition-era validation report.

In [7]:
identity = split['split_id'][:16]
split_path = ROOT / 'data/processed' / SOURCE / PACKAGE / f'split_{identity}.json'
audit_path = ROOT / 'experiments/results/overview' / SOURCE / PACKAGE / f'audit_{identity}.json'
save_json(split_path, split)
save_json(audit_path, summary)
print('Saved split:', split_path.relative_to(ROOT))
print('Saved audit:', audit_path.relative_to(ROOT))

Saved split: data/processed/chemicalspace_new/package_00/split_f85552ea5b17e482.json
Saved audit: experiments/results/overview/chemicalspace_new/package_00/audit_f85552ea5b17e482.json


## Checkpoint
For the first Chemicalspace package expect 1,200 raw rows, 1,200 eligible unique structures, 100 held-out queries, a 1,000-molecule index and 100 unused structures.

For a 100K parent, do not make separate splits for different sizes: use `index_records(split, size)` on this same saved split. Next comes freshly generated Morgan fingerprints and controlled RDKit/NMSLIB distance tests, followed by recall and timing implementations. No claim about measured retrieval performance is made by this notebook.